# R38 · Review existing results on CPU

Choose a **CPU / standard Python runtime**, then **Run all**. This notebook fetches one pinned source commit, verifies the saved inputs, replays the existing PR123/PR121/PR35 predictions, and exports a small results ZIP. No GPU, Torch installation, camera-image download or model weights are needed.

All eight R38 requests concern development labels: **three replacements and five explicit exclusions**. All 137 reviewed training files and eligible R37 training supervision remain unchanged. A second GPU training pair would have no new annotation treatment. The R37/R38, frozen 666-front and historical 663-front scores remain separate; **PR35 stays selected**.

Changed scores under R38 describe a changed reference. A deleted missed row is not a recovered detection. The ZIP preserves original R37 row identities, including deleted rows. The inherited pending `new_000010:24` is retained exactly.


In [ ]:
import hashlib, json, os, subprocess, sys, tempfile
from pathlib import Path

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
REPLAY_ENV = {key: value for key, value in os.environ.items() if key not in ("PYTHONPATH", "PYTHONHOME")}
SOURCE_REVISION = 'd0cccb385633636d59211da915debb18097a0076'
CONTRACT_SHA256 = '21de76ae990473b07571a86e8110cba4e77db66d6ec570558cfd6cfb9697b605'
SOURCE_FILES_SHA256 = {'scripts/front_review_r38_replay.py': 'abf49d15a6aa434af6321cc40fee4046eaae672d249f11789e8d0df3e1e264ee', 'scripts/score_pr123_review.py': '944da5f1aec67f604a9de3a5e0497de2ad043bd9f353230ce494682d1e8d7317', 'scripts/evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'scripts/pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'scripts/diagnose_pose_quality.py': 'a3baa1285d1580ee44547f7c3d606f524f2f40616fed568885746d2014159953', 'scripts/pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'experiments/front-review-2026-11-26-r38-replay.json': '21de76ae990473b07571a86e8110cba4e77db66d6ec570558cfd6cfb9697b605'}
SOURCE_URL = os.environ.get("BAOMAI_R38_SOURCE_URL", "https://github.com/Siuuuers/dl.git")
WORK = Path(os.environ.get("BAOMAI_R38_WORK", str(Path("/content") / "baomai-r38-replay" if Path("/content").is_dir() else Path.cwd() / "baomai-r38-replay"))).expanduser().resolve()
REPO = WORK / "source"
OUTPUT = WORK / "replay"
ARCHIVE = WORK / "baomai-front-review-r38-replay-results.zip"

def git(*args, cwd=None):
    env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
    process = subprocess.run(["git", "-c", "filter.lfs.process=", "-c", "filter.lfs.smudge=", "-c", "filter.lfs.clean=", "-c", "filter.lfs.required=false", *args], cwd=cwd, env=env, text=True, capture_output=True)
    if process.returncode:
        raise RuntimeError(process.stderr.strip() or "Git source retrieval failed")
    return process.stdout.strip()

def verify_checkout(path):
    if git("rev-parse", "HEAD", cwd=path) != SOURCE_REVISION:
        raise RuntimeError("Source revision differs. Preserve this work folder and choose a new BAOMAI_R38_WORK path.")
    if git("status", "--porcelain", "--untracked-files=all", cwd=path):
        raise RuntimeError("Source checkout is dirty. Preserve its changes and choose a new BAOMAI_R38_WORK path.")
    for relative, expected in SOURCE_FILES_SHA256.items():
        file = path / relative
        if not file.is_file() or hashlib.sha256(file.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Pinned replay source hash differs: {relative}")

def assemble_source():
    WORK.mkdir(parents=True, exist_ok=True)
    if REPO.exists():
        verify_checkout(REPO)
    else:
        with tempfile.TemporaryDirectory(prefix=".r38-source-", dir=WORK) as temporary:
            staged = Path(temporary) / "source"
            staged.mkdir()
            git("init", "--quiet", cwd=staged)
            git("fetch", "--quiet", "--depth", "1", SOURCE_URL, SOURCE_REVISION, cwd=staged)
            git("checkout", "--quiet", "--detach", "FETCH_HEAD", cwd=staged)
            verify_checkout(staged)
            staged.replace(REPO)
    return REPO

assemble_source()
print(f"Verified R38 replay source {SOURCE_REVISION[:12]}. CPU only; no image/weight downloads or package installation.")


## 1 · Verify the review and cached inputs

Check source and file hashes, all 157 original split identities, the 137 unchanged training files, eligible R37 labels, the unchanged original 20-image/664-front trainer validation, and deletion-aware row maps. The replay verifies image **identity hashes recorded in the caches**, without loading the original image bytes. Original-pixel annotation review is documented separately.


In [ ]:
verify_checkout(REPO)
checked = subprocess.run(
    [sys.executable, "-S", str(REPO / "scripts/front_review_r38_replay.py"),
     "--root", str(REPO), "--verify-only", "--contract-sha256", CONTRACT_SHA256],
    text=True, capture_output=True, env=REPLAY_ENV,
)
if checked.returncode:
    raise RuntimeError(checked.stderr.strip())
INPUT_AUDIT = json.loads(checked.stdout)
print(f"Verified {INPUT_AUDIT['input_files_verified']} input files; all 137 reviewed training labels are unchanged.")
print("Eligible R37 training:", INPUT_AUDIT["eligible_r37_training_counts"])
print("Original trainer validation:", INPUT_AUDIT["original_trainer_validation"])
print("R37 development fronts:", INPUT_AUDIT["review_counts"]["r37"]["val"]["objects"])
print("R38 development fronts:", INPUT_AUDIT["review_counts"]["r38"]["val"]["objects"])


## 2 · Replay and export

Use the original fixed scoring rules: detector confidence 0.13, box matching IoU 0.50, endpoint confidence 0.50, and 5/10-pixel corner reports. The cached inference used ordinary NMS at 0.70. No threshold or model-head sweep runs. Replay must exactly reproduce the preserved audited reports before the ZIP is accepted.

Re-running clean, completed cells verifies and reuses the same results. Changed source or output files are preserved and rejected, so a review run cannot silently replace your edits.


In [ ]:
verify_checkout(REPO)
completed = subprocess.run(
    [sys.executable, "-S", str(REPO / "scripts/front_review_r38_replay.py"),
     "--root", str(REPO), "--source-revision", SOURCE_REVISION,
     "--contract-sha256", CONTRACT_SHA256, "--output", str(OUTPUT), "--archive", str(ARCHIVE)],
    text=True, capture_output=True, env=REPLAY_ENV,
)
if completed.returncode:
    raise RuntimeError(completed.stderr.strip())
RESULT = json.loads(completed.stdout)
COMPARISON = json.loads((OUTPUT / "comparison.json").read_text())
for name in ("frozen666", "historical663", "r37", "r38"):
    data = COMPARISON["references"][name]
    print(f"\n{name}")
    for role in ("reference", "candidate", "pr35"):
        model = data["models"][role]
        detection, quality = model["detection"], model["corner_quality"]
        pairs = [item["both_visible_objects_within_radius"] for item in quality["radii"]]
        print(f"  {role}: missed {detection['fn']}, unmatched {detection['fp']}, duplicates {detection['duplicate_fp']}; corner pairs 5/10 px {pairs}/{quality['both_visible_objects']}")
print(f"\nVerified ZIP: {ARCHIVE.name} ({RESULT['archive_size_bytes']:,} bytes)")
print("SHA256:", RESULT["archive_sha256"])


## 3 · Keep the small result ZIP

The ZIP contains the separate score reports, R37-to-R38 reference-change analysis, row maps, operations, input/context audit, replay contract and file hashes. It contains no model checkpoints or original photographs. Keep it with the two cropped-row review PDFs. This notebook does not train an R38 model or change the selected checkpoint.

Colab downloads the ZIP below. In another notebook environment, use the printed absolute path. Initial source retrieval needs Git/network access; a verified existing source checkout can be reused offline.


In [ ]:
print("Result file:", ARCHIVE)
if os.environ.get("BAOMAI_R38_DOWNLOAD", "1") == "1":
    try:
        from google.colab import files
    except ImportError:
        print("Open the result path above in your notebook file browser.")
    else:
        files.download(str(ARCHIVE))
